# Diamond recursive meta-feedback smoke (10 min, WikiText-103)

This tests the new recursive feedback architecture:

- combined bit/byte + subword/word + sentence tokenizer bus
- meta state with temporal +0/+1/+2/+3 slots
- C(256,2) and C(256,3) meta experts
- feedback re-enters every byte, word, and sentence tower layer
- meta-future conditioning of the byte/word/sentence heads
- causal first-word sentence cache

WikiText-103 is used from the NaNROUTER release; whole-document holdout.


In [ ]:
#@title GPU / TensorFlow
!nvidia-smi || true
import tensorflow as tf
print("TensorFlow", tf.__version__)
print("GPUs", tf.config.list_physical_devices("GPU"))


In [ ]:
#@title Clone meta-feedback branch
%cd /kaggle/working
!rm -rf nanlm_experimental
!git clone --branch master --depth 1 https://github.com/shemshallah/nanlm_experimental.git
%cd /kaggle/working/nanlm_experimental
!git log --oneline -1


In [ ]:
#@title WikiText-103 download + article split
!if [ ! -s /kaggle/working/wiki.train.raw ]; then \
  wget -q --show-progress -O /kaggle/working/wiki.train.raw \
    https://github.com/shemshallah/NaNROUTER/releases/download/wikitext103/wiki.train.raw; \
fi
!python diamond/colab/prepare_wikitext.py \
  --raw /kaggle/working/wiki.train.raw \
  --out /kaggle/working/corpus \
  --max-articles 4000 \
  --max-chars 20000000


In [ ]:
#@title 10-minute meta-feedback training
!STATE_DIR=/kaggle/working/meta_smoke \
 CORPUS_DIR=/kaggle/working/corpus \
 MODEL_SIZE=frontier BATCH=32 EVAL_BATCH=32 \
 EPOCHS=1 EXAMPLES=1000 CORPUS_CHARS=5000000 \
 TIME_BUDGET=10 TEST_SIZE=1024 \
 bash diamond/kaggle/train_kaggle.sh


In [ ]:
#@title Full metrics
!python diamond/colab/report_metrics.py /kaggle/working/meta_smoke/diamond_metrics.json
